# DASS walkthrough

1. Curate a training set with DASS on public data (IMDB → SST-2)
2. Inspect the selection (statistics and t-SNE/PCA plots)
3. Fine-tune and evaluate a RoBERTa sentiment classifier on the curated data
4. Inspect the processed Amazon category datasets (if you've built them)
5. Summarize the saved experiment results

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))  # make the `dass` package importable from notebooks/

from datasets import Dataset, load_dataset
from dass import DASS

## 1. Curate a training set

In [ ]:
def to_input_label(dataset, text_col):
    return Dataset.from_dict({"input": dataset[text_col], "label": dataset["label"]})

source_dataset = to_input_label(load_dataset("imdb", split="train").shuffle(seed=0).select(range(1000)), "text")
sst2 = load_dataset("stanfordnlp/sst2", split="train").shuffle(seed=0)
target_pool = to_input_label(sst2.select(range(500)), "sentence")       # pool to pick annotations from
target_val  = to_input_label(sst2.select(range(500, 700)), "sentence")  # held-out target data
target_test = to_input_label(sst2.select(range(700, 950)), "sentence")

dass = DASS(
    source=source_dataset,
    target=target_pool,
    model="all-MiniLM-L6-v2",
    budget=300,
    target_ratio=0.3,
    source_method="fl1mi",
    source_eta=1.0,
)
train_dataset = dass.curate()
train_dataset

## 2. Inspect the selection

In [ ]:
stats = dass.generate_statistics(output_dir="dass_output/stats")
{k: v for k, v in stats.items()}

In [ ]:
from IPython.display import Image, display
display(Image("dass_output/stats/tsne_visualization.png", width=600))

## 3. Fine-tune and evaluate (GPU recommended)

In [ ]:
from dass.sentiment_classifier import SentimentClassifier

classifier = SentimentClassifier()
classifier.prepare_data(train_data=train_dataset, dev_data=target_val)
classifier.train(output_dir="dass_output/model_training", experiment_name="imdb-to-sst2", epochs=3, lr=2e-5)

results = classifier.evaluate(test_data=target_test, output_dir="dass_output/results")
classifier.visualize(output_dir="dass_output/results")
{k: results[k] for k in ["accuracy", "f1_score", "precision", "recall"]}

## 4. Processed Amazon category datasets

Built by `data/convert_to_hf_data.py` (first 1,000 reviews per category) and
`data/create_balanced_dataset.py` (label-balanced and de-duplicated).

In [ ]:
from collections import Counter
from datasets import load_from_disk

for name in ["final_dataset", "final_dataset_balanced"]:
    path = f"../data/processed/{name}"
    if not os.path.exists(path):
        print(f"{path} not found; see data/README.md"); continue
    ds = load_from_disk(path)
    print(name, {split: dict(Counter(ds[split]["label"])) for split in ds})

## 5. Saved experiment results

In [ ]:
!cd .. && python scripts/summarize_results.py